In [1]:
import json
import joblib
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [3]:
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
DATA_DIR = PROJECT_ROOT / 'data'

print("Project root resolved to:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)

# Load the new balanced dataset
df = pd.read_csv(DATA_DIR / '../dataset/Loan_Default_Clean_Balanced_5050.csv')
print("Initial shape:", df.shape)

Project root resolved to: c:\Users\krish\Desktop\STProject
Data directory: c:\Users\krish\Desktop\STProject\data
Initial shape: (73278, 29)


In [4]:
# CRITICAL: Drop Target Leakage Columns
# These features are generated after loan approval and cannot be used to predict risk.
leakage_cols = ['rate_of_interest', 'Interest_rate_spread', 'Upfront_charges']
df = df.drop(columns=[c for c in leakage_cols if c in df.columns])

# Separate features and target
X = df.drop('Status', axis=1)
y = df['Status']

print("X shape:", X.shape, "y shape:", y.shape)
print("Target distribution (%):\n", y.value_counts(normalize=True) * 100)

X shape: (73278, 25) y shape: (73278,)
Target distribution (%):
 Status
0    50.0
1    50.0
Name: proportion, dtype: float64


In [5]:
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

# Fill missing numerical values with the column Median
for col in numeric_cols:
    X[col] = X[col].fillna(X[col].median())

# Fill missing categorical values with the column Mode (most frequent)
for col in categorical_cols:
    X[col] = X[col].fillna(X[col].mode()[0])

print("Missing values remaining after imputation:", X.isnull().sum().sum())

Missing values remaining after imputation: 0


In [6]:
# Use OneHotEncoder for all categorical columns to handle the text variance
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoded_array = encoder.fit_transform(X[categorical_cols])
encoded_cols = encoder.get_feature_names_out(categorical_cols)

encoded_df = pd.DataFrame(encoded_array, columns=encoded_cols, index=X.index)

# Drop original categorical columns and concatenate the encoded ones
X = X.drop(columns=categorical_cols)
X = pd.concat([X, encoded_df], axis=1)

print("Shape after one-hot encoding:", X.shape)
X.head()

Shape after one-hot encoding: (73278, 60)


,loan_amount,term,property_value,income,Credit_Score,LTV,dtir1,loan_limit_cf,loan_limit_ncf,Gender_Female,...,age_55-64,age_65-74,age_<25,age_>74,submission_of_application_not_inst,submission_of_application_to_inst,Region_North,Region_North-East,Region_central,Region_south
0,526500,360.0,668000.0,11580.0,796,78.817365,22.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1,396500,300.0,398000.0,4260.0,854,76.162791,40.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,506500,360.0,398000.0,0.0,803,76.162791,40.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
3,56500,360.0,398000.0,960.0,714,76.162791,40.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4,166500,300.0,398000.0,2340.0,525,76.162791,40.0,1.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0


In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train default rate:", y_train.mean().round(4))
print("Test default rate:", y_test.mean().round(4))

Train shape: (58622, 60) Test shape: (14656, 60)
Train default rate: 0.5
Test default rate: 0.5


In [8]:
scaler = StandardScaler()

# Fit only on training data, transform both
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

X_train[numeric_cols].describe().round(2).transpose()

,count,mean,std,min,25%,50%,75%,max
loan_amount,58622.0,-0.0,1.0,-1.61,-0.73,-0.21,0.52,15.62
term,58622.0,0.0,1.0,-4.11,0.43,0.43,0.43,0.43
property_value,58622.0,-0.0,1.0,-1.33,-0.52,-0.21,0.22,46.03
income,58622.0,0.0,1.0,-1.06,-0.49,-0.19,0.22,59.61
Credit_Score,58622.0,0.0,1.0,-1.72,-0.87,-0.00,0.88,1.72
LTV,58622.0,-0.0,1.0,-1.91,-0.19,0.05,0.26,202.13
dtir1,58622.0,0.0,1.0,-3.50,-0.28,0.14,0.55,2.32


In [9]:
MODELS_DIR = PROJECT_ROOT
joblib.dump(scaler, MODELS_DIR / 'scaler.pkl')
joblib.dump(encoder, MODELS_DIR / 'encoder.pkl')

feature_columns = X_train.columns.tolist()
with open(MODELS_DIR / 'feature_names.json', 'w') as f:
    json.dump(feature_columns, f, indent=2)

print("Saved scaler.pkl, encoder.pkl, feature_names.json to:", MODELS_DIR)
print("Total features:", len(feature_columns))

Saved scaler.pkl, encoder.pkl, feature_names.json to: c:\Users\krish\Desktop\STProject
Total features: 60


In [11]:
X_train.to_csv(DATA_DIR / '../dataset/X_train.csv', index=False)
X_test.to_csv(DATA_DIR / '../dataset/X_test.csv', index=False)
y_train.to_csv(DATA_DIR / '../dataset/y_train.csv', index=False)
y_test.to_csv(DATA_DIR / '../dataset/y_test.csv', index=False)

print("Processed train/test splits saved to:", DATA_DIR)
print("Confirm files exist:", list(DATA_DIR.glob('*.csv')))

Processed train/test splits saved to: c:\Users\krish\Desktop\STProject\data
Confirm files exist: []
